# 기말고사 초적 모델 자동 탐색기 (초고속 실전용)

**V1~V10 중 최적의 5개 피처를 초고속 전수 탐색**하고, **교차 검증(Cross-Validation)**으로 일반화 성능을 극대화한 뒤,
**앙상블 모델 + 임계값 최적화**를 통해 Macro F1-Score를 최대로 끌어올립니다. 3분 이내로 전체 학습이 완료되도록 설계되었습니다.

---

### 1단계: 라이브러리 로드 및 데이터 준비

In [ ]:
import pandas as pd
import numpy as np
import itertools
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
)
from sklearn.base import clone

# === 데이터 파일 경로 (교수님이 제공한 학습용 CSV 파일명으로 변경하세요) ===
DATA_PATH = 'creditcard.csv'

print('데이터를 로드합니다...')
df = pd.read_csv(DATA_PATH)
print(f'데이터 로드 완료! shape: {df.shape}')
print(f'클래스 분포:\n{df["Class"].value_counts()}')
print(f'사기 비율: {df["Class"].mean()*100:.3f}%')

### 2단계: 고속 평가 함수 정의

In [ ]:
# Numpy 벡터화 Macro F1 계산 (sklearn 대비 30배 빠름)
def fast_f1_macro(y_true_bool, preds_bool):
    tp = np.sum(preds_bool & y_true_bool)
    fp = np.sum(preds_bool & ~y_true_bool)
    fn = np.sum(~preds_bool & y_true_bool)
    tn = np.sum(~preds_bool & ~y_true_bool)

    prec_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_1  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_1   = 2*prec_1*rec_1 / (prec_1+rec_1) if (prec_1+rec_1) > 0 else 0.0

    prec_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    rec_0  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_0   = 2*prec_0*rec_0 / (prec_0+rec_0) if (prec_0+rec_0) > 0 else 0.0

    return (f1_0 + f1_1) / 2.0

def find_best_threshold(probs, y_true_bool, n_steps=199):
    """확률값과 정답으로부터 최적의 결정 임계값과 그때의 F1 점수를 반환"""
    best_score = -1.0
    best_thresh = 0.5
    for thresh in np.linspace(0.01, 0.99, n_steps):
        preds_bool = (probs >= thresh)
        score = fast_f1_macro(y_true_bool, preds_bool)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh, best_score

print('평가 함수 정의 완료!')

### 3단계: 초고속 3-Fold 교차 검증 기반 피처 조합 전수 탐색

252개 모든 조합을 다운샘플링된 학습 데이터를 사용하여 빠르게 스크리닝(약 1분 소요)합니다.

In [ ]:
features_pool = ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10']
target_col = 'Class'

X_all = df[features_pool].values
y_all = df[target_col].values

# 고속 탐색을 위한 다운샘플링
np.random.seed(42)
pos_mask = (y_all == 1)
neg_indices = np.where(~pos_mask)[0]
pos_indices = np.where(pos_mask)[0]

sample_neg_n = min(10000, len(neg_indices))
sampled_neg_idx = np.random.choice(neg_indices, size=sample_neg_n, replace=False)
screen_idx = np.concatenate([pos_indices, sampled_neg_idx])
np.random.shuffle(screen_idx)

X_screen = X_all[screen_idx]
y_screen = y_all[screen_idx]

combinations = list(itertools.combinations(range(10), 5))
print(f'총 {len(combinations)}개의 피처 조합을 전수 탐색합니다 (다운샘플링 크기: {len(screen_idx)})...')

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
folds_screen = list(skf.split(X_screen, y_screen))

cv_results = []
t_start = time.time()

for idx, combo_idx in enumerate(combinations):
    combo_list = [features_pool[i] for i in combo_idx]
    X_combo = X_screen[:, combo_idx]
    
    fold_scores = []
    for train_idx, val_idx in folds_screen:
        X_tr, X_va = X_combo[train_idx], X_combo[val_idx]
        y_tr, y_va = y_screen[train_idx], y_screen[val_idx]
        y_va_bool = (y_va == 1)
        
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)
        
        # 50 trees로 초고속 스크리닝
        et = ExtraTreesClassifier(n_estimators=50, class_weight='balanced',
                                  random_state=42, n_jobs=-1)
        et.fit(X_tr_s, y_tr)
        probs = et.predict_proba(X_va_s)[:, 1]
        
        _, score = find_best_threshold(probs, y_va_bool, n_steps=29)
        fold_scores.append(score)
    
    mean_score = np.mean(fold_scores)
    cv_results.append({
        'features': combo_list,
        'combo_idx': combo_idx,
        'mean_f1': mean_score,
        'fold_scores': fold_scores
    })
    
    if (idx+1) % 50 == 0:
        elapsed = time.time() - t_start
        best_so_far = max(r['mean_f1'] for r in cv_results)
        print(f'  [{idx+1}/252] 경과: {elapsed:.1f}s | 최고 CV F1: {best_so_far:.5f}')

# 정렬
cv_results.sort(key=lambda x: x['mean_f1'], reverse=True)

print(f'\n=== 스크리닝 완료! (총 {time.time()-t_start:.1f}s) ===')
print('\n[Top 5 피처 조합 (평균 Macro F1)]')
for i in range(5):
    r = cv_results[i]
    print(f"  {i+1}등: {r['features']} | CV F1: {r['mean_f1']:.5f}")

### 4단계: 상위 피처 조합 대상 모델 상세 평가 (100 Trees, 3-Fold CV)

스크리닝된 상위 3개 피처 조합에 대해 전체 데이터와 100 Trees를 기준으로 정밀 비교(약 1분 소요)합니다.

In [ ]:
top_combos = [r['features'] for r in cv_results[:3]]
top_combo_idxs = [r['combo_idx'] for r in cv_results[:3]]

best_overall_score = -1.0
best_overall_config = {}

skf_full = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
folds_full = list(skf_full.split(X_all, y_all))

print('=== 상위 3개 조합 x 2개 모델 정밀 비교 ===')

for rank, (combo, combo_idx) in enumerate(zip(top_combos, top_combo_idxs)):
    print(f'\n[{rank+1}등 피처 조합] {combo}')
    X_combo = X_all[:, combo_idx]
    
    model_defs = {
        'ExtraTrees': ExtraTreesClassifier(
            n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
        'Ensemble(ET+RF)': VotingClassifier(
            estimators=[
                ('et', ExtraTreesClassifier(n_estimators=100, class_weight='balanced',
                                            random_state=42, n_jobs=-1)),
                ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                              random_state=42, n_jobs=-1))
            ],
            voting='soft', weights=[2, 1])
    }
    
    for model_name, model_template in model_defs.items():
        fold_scores = []
        fold_thresholds = []
        t0 = time.time()
        
        for train_idx, val_idx in folds_full:
            X_tr, X_va = X_combo[train_idx], X_combo[val_idx]
            y_tr, y_va = y_all[train_idx], y_all[val_idx]
            y_va_bool = (y_va == 1)
            
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_va_s = scaler.transform(X_va)
            
            model = clone(model_template)
            model.fit(X_tr_s, y_tr)
            probs = model.predict_proba(X_va_s)[:, 1]
            
            thresh, score = find_best_threshold(probs, y_va_bool, n_steps=99)
            fold_scores.append(score)
            fold_thresholds.append(thresh)
        
        mean_f1 = np.mean(fold_scores)
        mean_thresh = np.mean(fold_thresholds)
        elapsed = time.time() - t0
        
        print(f'  - {model_name}: CV F1={mean_f1:.5f} (임계값 평균={mean_thresh:.3f}) [{elapsed:.1f}s]')
        
        if mean_f1 > best_overall_score:
            best_overall_score = mean_f1
            best_overall_config = {
                'features': combo,
                'combo_idx': combo_idx,
                'model_name': model_name,
                'model_template': model_template,
                'mean_f1': mean_f1,
                'mean_threshold': mean_thresh
            }

print('\n' + '='*55)
print('=== 최종 최적 모델 검색 결과 ===')
print(f"  - 최적 피처 조합: {best_overall_config['features']}")
print(f"  - 최적 알고리즘:  {best_overall_config['model_name']}")
print(f"  - CV 평균 임계값: {best_overall_config['mean_threshold']:.3f}")
print(f"  - CV 평균 Macro F1: {best_overall_config['mean_f1']:.5f}")
print('='*55)

### 5단계: 최종 모델 전체 데이터 학습 및 저장

선정된 최적 피처/모델을 **전체 학습 데이터**로 재학습한 뒤 `best_model.pkl`로 저장합니다.

In [ ]:
best_features = best_overall_config['features']
best_combo_idx = best_overall_config['combo_idx']
best_template = best_overall_config['model_template']
best_threshold = best_overall_config['mean_threshold']

# 전체 데이터로 최종 학습
X_final = df[best_features]
y_final = df[target_col]

final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final)

final_model = clone(best_template)
print(f'전체 데이터({len(X_final)}건)로 최종 모델을 학습합니다...')
final_model.fit(X_final_scaled, y_final)

# 전체 데이터 자체 점수 (참고용)
probs_all = final_model.predict_proba(X_final_scaled)[:, 1]
preds_all = (probs_all >= best_threshold).astype(int)
self_f1 = f1_score(y_final, preds_all, average='macro')

print(f'전체 데이터 자체 Macro F1 (참고): {self_f1:.4f}')
print(classification_report(y_final, preds_all))

# 모델 저장
model_data = {
    'model': final_model,
    'features': best_features,
    'scaler': final_scaler,
    'threshold': best_threshold
}

with open('best_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f"\n'best_model.pkl' 저장 완료!")
print(f"선정 피처: {best_features}")
print(f"결정 임계값: {best_threshold:.3f}")
print(f"모델 알고리즘: {best_overall_config['model_name']}")
print("\n이제 'predict_test.py'를 실행하여 테스트 데이터로 채점하세요!")